# CheXpert-Small Google Collab Demonstration

This is a demonstration on how to train  

## Required libraries

### Install required libraries if needed.

In [1]:
import ipywidgets as widgets
from IPython.display import display

install_requirements = widgets.Dropdown(
    options=['yes', 'no'],
    value='no',
    description='Do you need to install required libraries?',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='auto', margin='0')
)

python_lib_framework = widgets.Dropdown(
    options=['conda', 'pip'],
    value='pip',
    description='What python package manager are you using?',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='auto', margin='0')
)

display(install_requirements)
display(python_lib_framework)

# Access the selected drop-down value later with:
# dropdown-variable_name.value


Dropdown(description='Do you need to install required libraries?', index=1, layout=Layout(margin='0', width='a…

Dropdown(description='What python package manager are you using?', index=1, layout=Layout(margin='0', width='a…

In [2]:
# Install requirements?
if(install_requirements.value == 'yes'):
    if python_lib_framework.value == 'conda':
        ! conda install -y tqdm
        ! conda install -y timm
        ! conda install -y numpy
        ! conda install -y pandas
        ! conda install -y pillow
        ! conda install -y pytorch
        ! conda install -y torchvision
        ! conda install -y tensorboard
    elif python_lib_framework.value == 'pip':
        ! pip install -y tqdm
        ! pip install -y timm
        ! pip install -y numpy
        ! pip install -y pandas
        ! pip install -y pillow
        ! pip install -y pytorch
        ! pip install -y torchvision
        ! pip install -y tensorboard
    else:
        raise ValueError("Invalid Python Library/Package Manager")
    
        

### Import libraries

In [3]:
import os 
import tqdm
import timm 
import yaml
import numpy as np
import pandas as pd
from PIL import Image
import torch
import argparse
from torch.utils import tensorboard
from torchvision.transforms import v2 
from torch.nn.parallel import DistributedDataParallel as DDP


## ChexPert Dataset Class

In [4]:
class CheXpertDataset(torch.utils.data.Dataset):
    def __init__(
            self, 
            root: str,
            split: str, 
            transform: (v2.Compose | None), 
            uncertainty: str = 'zeros'
        ) -> None:

        self.root = root

        # Loads the CSV file that contains image paths along 
        # with their corresponding annotation
        csv_path = os.path.join(root, f'{split}.csv')
        self.df = pd.read_csv(csv_path)

        # Specify augmentations 
        self.transform = transform

        # Specify how to handle 'uncertain' annotations 
        # Default is to make these values 0.0 which is 'not present in scan'
        self.uncertainty = uncertainty

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]: 
        sample = self.df.iloc[idx]

        # Load image path from CSV row and load image using PIL 
        sample_path = str(sample['Path'])
        sample_path = sample_path.replace('CheXpert-v1.0/', '')
        sample_path = os.path.join(self.root, sample_path)
    
        # Load image
        image = Image.open(sample_path).convert('L')

        # Augment image, if applicable
        if self.transform:
            image = self.transform(image)

        # Extract labels from CSV row 
        labels = sample[5:].values.astype(np.float32)

        # Handle NaN values and uncertainty
        if self.uncertainty == 'zeros':
            labels = np.nan_to_num(labels, nan=0.0)
            labels = np.where(labels == -1.0, 0.0, labels)

        elif self.uncertainty == 'ones':
            labels = np.nan_to_num(labels, nan=0.0)
            labels = np.where(labels == -1.0, 1.0, labels)

        elif self.uncertainty == 'ignore':
            labels = np.nan_to_num(labels, nan=0.0)

        labels = torch.tensor(labels)

        return image, labels

In [5]:

"""
Training run on GPU process 
"""
def train(cpu_threads: int, is_logger: bool, params: dict) -> None: 
    """Model training routine.

    Parameters
    ----------
    cpu_threads : int
        How many threads to use by DataLoader class when creating batches.
    is_logger : bool
        When True, enables Tensorboard logs and progress bar.
    params : dict
        Configuration of training session.

    Returns
    -------
    None
        No return value.
    """

    # Specify GPU [Different from HPC version]
    device = "cuda" if torch.cuda.is_available() else "cpu"


    # Load parameters specified in config file 
    model_name = params['model_name']
    log_dir = params['log_dir']
    data_root = params['data']['root']
    batch_size = int(params['data']['batch_size'])
    num_epochs = int(params['optimization']['num_epochs'])
    learning_rate = float(params['optimization']['learning_rate'])

    # Setup Tensorboard logging 
    if is_logger:
        os.makedirs(log_dir, exist_ok=True)
        writer = tensorboard.SummaryWriter(log_dir=log_dir)

    """
    Initialize and setup model 
    """
    # Load from timm
    model = timm.create_model(model_name, pretrained=False)

    # Change the first conv. layer to 1 channel for greyscale input 
    model.conv1 = torch.nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

    # Change the final FC layer to output 14 classes to fit CheXpert 
    model.fc = torch.nn.Linear(model.fc.in_features, 14)

    # Put model on GPU and wrap in DDP for distributed training
    model.to(device)
    
    # Removed for Colab
    # model = DDP(model, device_ids=[local_rank])

    """
    Build dataset
    Setup the following for train and val. splits:
        - Augmentations 
        - Dataset
        - DataLoader 
    """
    train_transform = v2.Compose([
        v2.Resize((224, 224)),
        v2.RandomHorizontalFlip(),
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=[0.53305], std=[0.03491])
    ])

    val_transform = v2.Compose([
        v2.Resize((288, 288)),
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=[0.53305], std=[0.03491])
    ])

    train_dataset = CheXpertDataset(
        root=data_root,
        split='train',
        transform=train_transform
    )

    val_dataset = CheXpertDataset(
        root=data_root,
        split='valid',
        transform=val_transform
    )

    # [Different from HPC code.]
    # num_workers=0 avoids broken pipe errors in notebook/Colab environments
    # (multiprocessing with notebook-defined classes causes pickling failures)
    train_dataloader = torch.utils.data.DataLoader(
        dataset=train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=(device == "cuda"),
    )

    val_dataloader = torch.utils.data.DataLoader(
        dataset=val_dataset, 
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=(device == "cuda"),
    )  

    # Initialize optimizer 
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate
    )

    # Initialize loss function  
    loss_fn = torch.nn.BCEWithLogitsLoss()

    # Start training loop 
    for e in range(num_epochs): 

        # [Not the same as HPC code]        
        # This is for the sampler used by the DataLoader to randomize the 
        # order of the data in every epoch 
        #sampler.set_epoch(e)

        # Training epoch 
        model.train()
        pbar = tqdm.tqdm(train_dataloader) if is_logger else train_dataloader
        for idx, (img, lbl) in enumerate(pbar): 

            # Clear optimizer for current step 
            optimizer.zero_grad()

            # Move data to GPU 
            img = img.to(device)
            lbl = lbl.to(device)

            # Forward pass 
            out = model(img)
            
            # Backprop
            loss = loss_fn(out, lbl)
            loss.backward()
            optimizer.step()

            # Update progress bar and add loss to Tensorboard 
            if is_logger: 
                pbar.update(1)

                itr = e * len(train_dataloader) + idx
                writer.add_scalar('Loss/train', loss.item(), itr)

        # Validation epoch
        # Same as training epoch just no backprop
        model.eval()
        pbar = tqdm.tqdm(val_dataloader) if is_logger else val_dataloader
        for idx, (img, lbl) in enumerate(pbar): 
            img = img.to(device)
            lbl = lbl.to(device)

            out = model(img)

            loss = loss_fn(out, lbl)

            if is_logger: 
                pbar.update(1)

                itr = e * len(train_dataloader) + idx
                writer.add_scalar('Loss/validation', loss.item(), itr)


## Run training demonstration

In [ ]:
## Load configuration file
config_file_path = '../config/colab_config.yaml'
with open(config_file_path, 'r') as f:
    params = yaml.load(f, Loader=yaml.FullLoader)

# Train model
train(4, True, params)

  7%|▋         | 41/625 [48:47<13:06:31, 80.81s/it]